# Consolidate Dataset
Combining bug reports dataset from 5 datasets in a standard format and storing in parquet format for using in experiments,

Dataset information:
1) Bench4BL (2018)
2) BeetleBox (2024)
3) Long Code Arena (2023)
4) SWE-Bench (2024)
5) Ye et al. (2015)

In [6]:
import pandas as pd
import os
from datasets import load_from_disk
import re
import ast

/home/cs21d002_eashaan/PhD/Objective1/obj1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration
Loading all dataset and metadata information

In [ ]:
# load either path to the dataset or csv and store in pandas dataframe

# Step 1: Load the final project metadata CSV
project_metadata_df = pd.read_csv('/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/project_metadata.csv')

# Step 2: Load the 5 bug localization benchmark datasets

# a) Ye et al. dataset
ye_dataframe = pd.read_csv('/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/ye_et_al_full_data.csv')

# b) Bench4BL dataset
bench4bl_dataframe = pd.read_csv('/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/bench4bl_full_data.csv')

# c) LCA Dataset
# Path to the main LCA dataset file 
lca_py_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/py/dev-00000-of-00001.parquet'
lca_java_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/java/dev-00000-of-00001.parquet'
lca_kt_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/kt/dev-00000-of-00001.parquet'

# Load dev split into pandas dataframe
lca_py_dev = pd.read_parquet(lca_py_dev_file_path)
lca_java_dev = pd.read_parquet(lca_java_dev_file_path)
lca_kt_dev = pd.read_parquet(lca_kt_dev_file_path)

# Combine the python, java, kotlin dev dataframes into one dataframe
lca_dataframe = pd.concat([lca_py_dev, lca_java_dev, lca_kt_dev], ignore_index=True)

# d) BettleBox Dataset
# Path of the root directory of the BettleBox dataset
beetlebox_directory = "/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/BLAZE/15122980/Dataset/Dataset/BeetleBox"

# Load main bug beetlebox dataset
dataset = load_from_disk(beetlebox_directory)

# Combine train and test splits for a holistic overview
beetlebox_train = dataset['train'].to_pandas()
beetlebox_test = dataset['test'].to_pandas()

beetlebox_dataframe = pd.concat([beetlebox_train, beetlebox_test], ignore_index=True)

# e) SWE Bench Dataset
def extract_file_paths_from_patch(patch_text):
    '''
    Parses a patch string which is column in the dataset to extract unique file paths.

    Args:
        patch_text (str): The full text content of a patch.
    
    Returns:
        list: A list of unique file paths found in the patch.
    '''

    # We are creating a regex pattern which looks for lines starting with '--- a/' or '+++ b/'
    # and captuires the file path that follows. It also handles file paths that may contain spaces.
    regex = r"^(?:--- a\/|\+\+\+ b\/)(.+?)\t*$"

    # Find all matches in the text
    matches = re.findall(regex, patch_text, re.MULTILINE)

    # Return a list of unique file paths
    return list(set(matches))

# Path to the main SWE Bench dataset file 
swe_bench_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/dev-00000-of-00001.parquet'
swe_bench_train_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/train-00000-of-00001.parquet'
swe_bench_test_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/test-00000-of-00001.parquet'

# 3. Load all split into pandas dataframe
swe_bench_dev = pd.read_parquet(swe_bench_dev_file_path)
swe_bench_train = pd.read_parquet(swe_bench_train_file_path)
swe_bench_test = pd.read_parquet(swe_bench_test_file_path)

# Combine the train, dev, test dataframes into one dataframe
swe_bench_dataframe = pd.concat([swe_bench_train, swe_bench_dev, swe_bench_test], ignore_index=True)


 # Applying the function to the 'patch' column to get a list of files for each row
swe_bench_dataframe['ground_truth_files'] = swe_bench_dataframe['patch'].apply(extract_file_paths_from_patch)

## Process data from all five dataset dataframes

In [ ]:
def ensure_list_of_strings(data):
    '''
    Helper function to ensure the ground truth files are always a list of strings.
    '''
    if isinstance(data, list):
        return [str(item) for item in data]
    if isinstance(data, str):
        # Handle string-encoded list like "['file1', 'file2']"
        if data.startswith('[') and data.endswith(']'):
            try:
                evaluated = ast.literal_eval(data)
                if isinstance(evaluated, list):
                    return [str(item) for item in evaluated]
            except (ValueError, SyntaxError):
                return [data] # Return the string in a list if parsing fails
        
        # Handle delimited strings like whitespace between filenames like in ye et al. dataset "file1 file2"
        if ' ' in data:
            return data.split(' ')
        return [data]

    # For any other type or NaN, return an empty list
    return []    

In [31]:
def standardize_repo_name(name):
    '''
    Helper function to standardize repository names.
    '''
    if isinstance(name, str):
        return name.lower().strip()
    return name

In [65]:
def process_data(project_meta_df, ye_df, lca_df, beetlebox_df, bench4bl_df, swe_bench_df):
    '''
    Consolidates multiple bug report datasets into a single, clean Parquet file.
    '''
    # Standardize repo_name in the metadata file first
    project_meta_df['repo_name'] = project_meta_df['repo_name'].apply(standardize_repo_name)

    print("Standardizing columns for each dataset...")
    
    # Step 1: Standarized Ye et al. 
    ye_std = pd.DataFrame()
    ye_std['repo_name'] = ye_df['repo_name'].apply(standardize_repo_name)
    ye_std['bug_id'] = ye_df['bug_id'].astype(str)
    ye_std['bug_report_text'] = ye_df['summary'].fillna('') + '\n' + ye_df['description'].fillna('')
    # ye_std['ground_truth_files'] = ye_df['files'].apply(lambda x: x.split(' ') if isinstance(x, str) else [])
    ye_std['ground_truth_files'] = ye_df['files'].apply(ensure_list_of_strings)
    ye_std['creation_date'] = pd.to_datetime(ye_df['report_timestamp'], unit='ms', utc=True)
    ye_std['fix_date'] = pd.to_datetime(ye_df['commit_timestamp'], unit='ms', utc=True)
    ye_std['pre_fix_commit_sha'] = ye_df['before_fix_sha']
    ye_std['fix_commit_sha'] = ye_df['after_fix_sha']
    ye_std['language'] = ye_df['language']
    ye_std['source_dataset'] = 'Ye et al.'
    # Fix missing URL columns with None
    ye_std['bug_report_url'] = None
    ye_std['fix_url'] = None

    # Step 2: Standarized Long Code Arena (LCA) 
    lca_std = pd.DataFrame()
    lca_std['repo_name'] = (lca_df['repo_owner'] + '/' + lca_df['repo_name']).apply(standardize_repo_name)
    lca_std['bug_id'] = lca_df['id'].astype(str)
    lca_std['bug_report_url'] = lca_df['issue_url']
    lca_std['fix_url'] = lca_df['pull_url']
    lca_std['bug_report_text'] = lca_df['issue_title'].fillna('') + '\n' + lca_df['issue_body'].fillna('')
    # lca_std['ground_truth_files'] = lca_df['changed_files'].apply(ast.literal_eval)
    lca_std['ground_truth_files'] = lca_df['changed_files'].apply(ensure_list_of_strings)
    lca_std['creation_date'] = None  # Not directly available 
    lca_std['fix_date'] = pd.to_datetime(lca_df['pull_create_at'], utc=True)
    lca_std['pre_fix_commit_sha'] = lca_df['base_sha']
    lca_std['fix_commit_sha'] = lca_df['head_sha']
    lca_std['language'] = lca_df['repo_language']
    lca_std['source_dataset'] = 'Long Code Arena'

    # Step 3: Standarized BeetleBox 
    beetlebox_std = pd.DataFrame()
    beetlebox_std['repo_name'] = beetlebox_df['repo_name'].apply(standardize_repo_name)
    beetlebox_std['bug_id'] = beetlebox_df['issue_id'].astype(str)
    beetlebox_std['bug_report_url'] = beetlebox_df['issue_url']
    beetlebox_std['fix_url'] = beetlebox_df['pull_url']
    beetlebox_std['bug_report_text'] = beetlebox_df['title'].fillna('') + '\n' + beetlebox_df['body'].fillna('')
    # beetlebox_std['ground_truth_files'] = beetlebox_df['updated_files'].apply(ast.literal_eval)
    beetlebox_std['ground_truth_files'] = beetlebox_df['updated_files'].apply(ensure_list_of_strings)
    beetlebox_std['creation_date'] = pd.to_datetime(beetlebox_df['report_datetime'], utc=True) 
    beetlebox_std['fix_date'] = pd.to_datetime(beetlebox_df['commit_datetime'], utc=True)
    beetlebox_std['pre_fix_commit_sha'] = beetlebox_df['before_fix_sha']
    beetlebox_std['fix_commit_sha'] = beetlebox_df['after_fix_sha']
    beetlebox_std['language'] = beetlebox_df['language']
    beetlebox_std['source_dataset'] = 'BeetleBox'

    # Step 4: Standardize Bencn4BL
    bench4bl_std = pd.DataFrame()
    bench4bl_std['repo_name'] = bench4bl_df['repo_name'].apply(standardize_repo_name)
    bench4bl_std['bug_id'] = bench4bl_df['bug_id'].astype(str)
    bench4bl_std['bug_report_text'] = bench4bl_df['bug_report'].fillna('')
    # bench4bl_std['ground_truth_files'] = bench4bl_df['fixed_files']
    bench4bl_std['ground_truth_files'] = bench4bl_df['fixed_files'].apply(ensure_list_of_strings)
    bench4bl_std['creation_date'] = pd.to_datetime(bench4bl_df['report_date'], utc=True) 
    bench4bl_std['fix_date'] = pd.to_datetime(bench4bl_df['fix_date'], utc=True)
    bench4bl_std['pre_fix_commit_sha'] = bench4bl_df['before_fix_sha']
    bench4bl_std['language'] = bench4bl_df['language']
    bench4bl_std['source_dataset'] = 'Bench4BL'
    # Fill missing URL/SHA columns with None
    bench4bl_std['bug_report_url'] = None
    bench4bl_std['fix_url'] = None
    bench4bl_std['fix_commit_sha'] = None

    # Step 5: SWE-Bench
    swe_bench_std = pd.DataFrame()
    swe_bench_std['repo_name'] = swe_bench_df['repo'].apply(standardize_repo_name)
    swe_bench_std['bug_id'] = swe_bench_df['instance_id'].astype(str)
    swe_bench_std['bug_report_text'] = swe_bench_df['problem_statement'].fillna('') + '\n' + swe_bench_df['hints_text'].fillna('')
    # swe_bench_std['ground_truth_files'] = swe_bench_df['ground_truth_files']
    swe_bench_std['ground_truth_files'] = swe_bench_df['ground_truth_files'].apply(ensure_list_of_strings)
    swe_bench_std['fix_date'] = pd.to_datetime(swe_bench_df['created_at'], utc=True)
    swe_bench_std['pre_fix_commit_sha'] = swe_bench_df['base_commit']
    swe_bench_std['language'] = 'python'
    swe_bench_std['source_dataset'] = 'SWE-Bench'
    # Fill missing URL/SHA columns with None
    swe_bench_std['bug_report_url'] = None
    swe_bench_std['fix_url'] = None
    swe_bench_std['creation_date'] = None
    swe_bench_std['fix_commit_sha'] = None

    print("Concatenating all datasets....")
    # Step 6: Concatenate and Deduplicate
    all_bugs_df = pd.concat([ye_std, lca_std, beetlebox_std, bench4bl_std, swe_bench_std], ignore_index=True)

    print(f"Total bug reports before deduplication: {len(all_bugs_df)}")
    print(f"Total unique repos before filteration: {len(all_bugs_df['repo_name'].unique())}")

    # Implemented Hierarchical Deduplication based on different set of data available in these dataset

    # Making a copy of the original dtypes to restore later
    original_dtypes = all_bugs_df.dtypes

    # Level 1: Prioritize fix_url (Deduplicate on fix_url for rows that have it)
    with_url = all_bugs_df.dropna(subset=['fix_url'])
    without_url = all_bugs_df[all_bugs_df['fix_url'].isna()]
    with_url_deduped = with_url.drop_duplicates(subset=['fix_url'], keep='first')
    processed_df = pd.concat([with_url_deduped, without_url])
    
    # Level 2: Fallback to fix_commit_sha (deduplicate on fix_commit_sha for the remaining rows)
    with_sha = processed_df.dropna(subset=['fix_commit_sha'])
    without_sha = processed_df[processed_df['fix_commit_sha'].isna()]
    with_sha_deduped = with_sha.drop_duplicates(subset=['repo_name', 'fix_commit_sha'], keep='first')
    processed_df = pd.concat([with_sha_deduped, without_sha])


    # Level 3: Fallback to bug_id (Deduplicate on bug_id for the remaining rows)
    with_id = processed_df.dropna(subset=['bug_id'])
    without_id = processed_df[processed_df['bug_id'].isna()]
    with_id_deduped = with_id.drop_duplicates(subset=['repo_name', 'bug_id'], keep='first')
    processed_df = pd.concat([with_id_deduped, without_id])

    # Level 4: Final cleanup with bug_report_text
    final_deduped_df = processed_df.drop_duplicates(subset=['repo_name', 'bug_report_text'], keep='first')

    print(f"Total bug reports after deduplication: {len(final_deduped_df)}")

    # The final dataframe is now final_deduped_df
    all_bugs_df = final_deduped_df

    # Restore dtypes for columns that might have been converted to object due to NaN
    all_bugs_df = all_bugs_df.astype(original_dtypes, errors='ignore')

    # Step 7: Filter by Selected Projects
    final_project_list = project_meta_df['repo_name'].unique()
    final_bugs_df = all_bugs_df[all_bugs_df['repo_name'].isin(final_project_list)].copy()

    # Debugging Check
    metadata_repos = set(project_meta_df['repo_name'])
    final_bug_repos = set(final_bugs_df['repo_name'])
    missing_repos = metadata_repos - final_bug_repos

    if missing_repos:
        print(f"WARNING: {len(missing_repos)} repos from metadata are missing in the final bug report file.")
        print("This is likely due to repo name mismatches. Missing repos.")
        for repo in sorted(list(missing_repos)):
            print(f"-- {repo} --")
        print(" Diagnostic Check:  Bug Counts for MISSING Repos")
        # Filter the metadata to show only the projects that are missing
        missing_repos_metadata = project_meta_df[project_meta_df['repo_name'].isin(missing_repos)]
        # Display the expected bug counts for the missing repos
        print("Expected bug counts for missing repos (from metadata):")
        print(missing_repos_metadata[['repo_name', 'total_unique_bug_reports']].to_string())

    else:
        print("SUCCESS: All repositories from the metadata file are present in the final bug report file.")

    print(f"Total bug reports after filtering for selected projects: {len(final_bugs_df)}")
    print(f"Total unique repositories in final_bugs_df: {len(final_bugs_df['repo_name'].unique())}")

    print("-- Diagnostic Check: Bug Counts for MATCHED Repos")
    # Get the list of repos that were successfully processed
    matched_repos = set(final_bugs_df['repo_name'])
    matched_repos_metadata = project_meta_df[project_meta_df['repo_name'].isin(matched_repos)]
    # Calculate the final count of bug reports per repo in the processed dataframe
    final_counts = final_bugs_df.groupby('repo_name').size().reset_index(name='final_bug_count')

    # Merge the expected counts with the final counts for a side-by-side comparison
    comparison_df = pd.merge(
        matched_repos_metadata[['repo_name', 'total_unique_bug_reports']],
        final_counts,
        on='repo_name',
        how='left'
    )

    print("Comparison of expected vs. final bug counts for matched repos:")
    print(comparison_df.to_string())
    # comparison_df.to_csv('/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/comparison.csv')
    

    # Step 8: Create output directory and save files
    output_dir = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/data/processed/'
    os.makedirs(output_dir, exist_ok=True)

    # Save project metadata
    project_meta_df.to_parquet(os.path.join(output_dir, 'project_metadata.parquet'), index=False)
    print(f"Saved projects_metadata.parquet to {output_dir}")

    # Save bug reports
    final_bugs_df.to_parquet(os.path.join(output_dir, 'bug_reports.parquet'), index=False)
    print(f"Saved bug_reports.parquet to {output_dir}")

In [66]:
# Calling method and store data in parquet format
process_data(project_metadata_df, ye_dataframe, lca_dataframe, beetlebox_dataframe, bench4bl_dataframe, swe_bench_dataframe)

Standardizing columns for each dataset...
Concatenating all datasets....
Total bug reports before deduplication: 81850
Total unique repos before filteration: 974


/tmp/ipykernel_220778/2523077786.py:95: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_bugs_df = pd.concat([ye_std, lca_std, beetlebox_std, bench4bl_std, swe_bench_std], ignore_index=True)


Total bug reports after deduplication: 77996
SUCCESS: All repositories from the metadata file are present in the final bug report file.
Total bug reports after filtering for selected projects: 71452
Total unique repositories in final_bugs_df: 98
-- Diagnostic Check: Bug Counts for MATCHED Repos
Comparison of expected vs. final bug counts for matched repos:
                                       repo_name  total_unique_bug_reports  final_bug_count
0                                   apache/camel                      1419             1469
1                                   apache/hbase                       758              838
2                                    apache/hive                       971             1238
3                                wildfly/wildfly                       957              984
4                           wildfly/wildfly-core                       353              360
5                            apache/commons-math                       237              2

In [1]:
import pandas as pd

In [6]:
# Replacing the value 'high' with 'large' in column project_size in project_metadata.parquet 
df = pd.read_parquet('/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/data/processed/project_metadata.parquet')

# print(df.columns.tolist())

# Replace 'high' with 'large' in the 'project_size' column
df["project_size"] = df["project_size"].replace("high", "large")

# Save the updated DataFrame back to Parquet
df.to_parquet("/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/data/processed/project_metaadata.parquet", index=False)


In [7]:
df['project_size'].value_counts()

project_size
large     34
small     34
medium    30
Name: count, dtype: int64

In [2]:
df = pd.read_parquet('/home/cs21d002_eashaan/PhD/Objective1/data/processed/bug_reports.parquet')

In [10]:
import pandas as pd
import subprocess
from datetime import datetime

# Step 1: Filter for the target repo
target_repo = 'apache/hive'
owner_repo = target_repo.replace('/', '_')
repo_path = f'/home/cs21d002_eashaan/PhD/Objective1/data/repos/java/{owner_repo}'

df_repo = df[df['repo_name'] == target_repo].copy()

print("shape: ", df_repo.shape)

# Step 2: Drop rows with missing SHA
df_repo = df_repo[df_repo['pre_fix_commit_sha'].notna()]

print("shape after dropping missing sha row: ", df_repo.shape)

# Step 2: Function to get commit date from SHA
def get_commit_date(sha, repo_path):
    try:
        result = subprocess.run(
            ['git', '-C', repo_path, 'show', '-s', '--format=%ci', sha],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=True
        )
        date_str = result.stdout.strip()
        return datetime.strptime(date_str, '%Y-%m-%d %H:%M:%S %z').date()
    except Exception as e:
        print(f"Error fetching date for {sha}: {e}")
        return None

# Step 3: Extract commit dates
df_repo['commit_date'] = df_repo['pre_fix_commit_sha'].apply(lambda sha: get_commit_date(sha, repo_path))

# Step 4: Filter for year 2013
df_2008 = df_repo[df_repo['commit_date'].apply(lambda d: d and d.year == 2008)]

# Step 5: Show ground truth files from one matching bug report
if not df_2008.empty:
    sample_files = df_2008.iloc[3]['bug_report_text']
    print("bug_Report_text:", sample_files)
else:
    print("No bug reports found for 2008 in the specified repo.")


shape:  (1238, 12)
shape after dropping missing sha row:  (1238, 12)
bug_Report_text: better error code from Hive describe command
cryptic, non-informative error message
hive&gt; describe hive1_scribeloadertest
FAILED: Execution Error, return code 1 from org.apache.hadoop.hive.ql.exec.DDLTask
in this case the table was missing. better say that.


In [ ]:
df_repo = df[df['repo_name'] == 'apache/hive'].copy()